# 03 — Il confronto a parità di genere (Versione 3)

Tutti i modelli finiti, per testata, sulle 304 testate valutate. Metrica principale (fissata prima): **media sugli
orientamenti** dell'accuratezza bilanciata; poi l'AUC media sugli orientamenti, il dettaglio per orientamento e le
metriche su tutto. Per ModernBERT con più seed c'è l'**ensemble** (media delle probabilità, deciso prima).

Intervalli al 95% con un **bootstrap per gruppo** (2000 ricampionamenti) e differenze con `feature_genere_tuned`
(se c'è, altrimenti `feature_genere`) sugli stessi ricampionamenti. Gira da solo dopo ogni esperimento della coda.

In [1]:
import json
import re

import numpy as np
import pandas as pd

from comune import RESULTS, VERSIONI, metriche, per_testata

modelli = {}
for d in sorted(RESULTS.iterdir()):
    if d.is_dir() and d.name != "transformer" and (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
seed = {}
for n in list(modelli):
    m = re.match(r"(.+)_s\d+$", n)
    if m:
        seed.setdefault(m[1], []).append(n)
for config, nomi in seed.items():
    if len(nomi) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi]
        e = parti[0][["domain", "fold", "y", "orientamento"]].copy(); e["p"] = sum(p.loc[e.index, "p"] for p in parti) / len(parti)
        modelli[f"{config}_ensemble{len(nomi)}"] = e.reset_index()
# la combinazione fissata la sera del 20 ottobre (dopo aver visto il seed 42, prima degli altri seed): media 0,5/0,5
# delle probabilità del TF-IDF di riferimento e dell'ensemble di ModernBERT-base (tutti i seed finiti)
ens = [n for n in modelli if n.startswith("modernbert_base_genere_ensemble")]
if "feature_genere_tuned" in modelli and ens:
    a, b = modelli["feature_genere_tuned"].set_index("article_id"), modelli[ens[0]].set_index("article_id")
    e = a[["domain", "fold", "y", "orientamento"]].copy(); e["p"] = 0.5 * a.p + 0.5 * b.loc[a.index, "p"]
    modelli["combinazione_tfidf_modernbert"] = e.reset_index()
print("modelli finiti:", list(modelli))
T = {n: per_testata(p) for n, p in modelli.items()}
tutto = pd.DataFrame({n: metriche(t) for n, t in T.items()}).T
tutto.to_csv(RESULTS / "03_genere_metriche.csv")
tutto.round(3)

modelli finiti: ['bge_genere', 'e5_genere', 'feature_genere', 'feature_genere_senza_nomi', 'feature_genere_solo_funzione', 'feature_genere_tuned', 'modernbert_base_genere_s1', 'modernbert_base_genere_s42', 'modernbert_base_genere_ensemble2', 'combinazione_tfidf_modernbert']


,testate,media sugli orientamenti,AUC media sugli orientamenti,bilanciata (nessuno),bilanciata Far Left,bilanciata Far Right,bilanciata Slightly Left,bilanciata Slightly Right,bilanciata su tutto,accuratezza,F1 macro
bge_genere,304.0,0.502,0.593,0.735,0.375,0.429,0.395,0.575,0.657,0.664,0.658
e5_genere,304.0,0.569,0.611,0.755,0.500,0.593,0.395,0.601,0.692,0.697,0.693
feature_genere,304.0,0.618,0.671,0.707,0.625,0.650,0.544,0.562,0.703,0.714,0.704
feature_genere_senza_nomi,304.0,0.589,0.656,0.700,0.625,0.650,0.447,0.523,0.683,0.694,0.684
feature_genere_solo_funzione,304.0,0.618,0.610,0.700,0.625,0.636,0.531,0.599,0.683,0.697,0.683
feature_genere_tuned,304.0,0.593,0.735,0.766,0.625,0.600,0.461,0.511,0.707,0.717,0.709
modernbert_base_genere_s1,304.0,0.533,0.639,0.636,0.500,0.579,0.382,0.568,0.691,0.688,0.687
modernbert_base_genere_s42,304.0,0.579,0.632,0.684,0.500,0.521,0.478,0.709,0.709,0.707,0.706
modernbert_base_genere_ensemble2,304.0,0.561,0.633,0.672,0.542,0.550,0.421,0.618,0.706,0.707,0.705
combinazione_tfidf_modernbert,304.0,0.590,0.683,0.711,0.625,0.564,0.461,0.591,0.735,0.740,0.736


In [2]:
# per orientamento, quante testate e il dettaglio di ogni modello
print(T[next(iter(T))].groupby("orientamento").y.agg(testate="size", affidabili="sum"))
tutto[[c for c in tutto.columns if c.startswith("bilanciata")]].round(2)

                testate  affidabili
orientamento                       
(nessuno)            82          64
Far Left             16          12
Far Right            84          14
Slightly Left        44          38
Slightly Right       78          40


,bilanciata (nessuno),bilanciata Far Left,bilanciata Far Right,bilanciata Slightly Left,bilanciata Slightly Right,bilanciata su tutto
bge_genere,0.74,0.38,0.43,0.39,0.57,0.66
e5_genere,0.76,0.50,0.59,0.39,0.60,0.69
feature_genere,0.71,0.62,0.65,0.54,0.56,0.70
feature_genere_senza_nomi,0.70,0.62,0.65,0.45,0.52,0.68
feature_genere_solo_funzione,0.70,0.62,0.64,0.53,0.60,0.68
feature_genere_tuned,0.77,0.62,0.60,0.46,0.51,0.71
modernbert_base_genere_s1,0.64,0.50,0.58,0.38,0.57,0.69
modernbert_base_genere_s42,0.68,0.50,0.52,0.48,0.71,0.71
modernbert_base_genere_ensemble2,0.67,0.54,0.55,0.42,0.62,0.71
combinazione_tfidf_modernbert,0.71,0.62,0.56,0.46,0.59,0.74


In [3]:
B = 2000
rif = "feature_genere_tuned" if "feature_genere_tuned" in T else "feature_genere"
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo
dom = T[rif].index; g = gruppo.reindex(dom).to_numpy(); gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def principale(t, idx):
    return metriche(t.iloc[idx])["media sugli orientamenti"]


boot = {n: np.array([principale(t.reindex(dom), idx) for idx in campioni]) for n, t in T.items()}
righe = {}
for n in T:
    b = boot[n]
    r = {"media sugli orientamenti": tutto.loc[n, "media sugli orientamenti"], "2.5%": np.nanpercentile(b, 2.5), "97.5%": np.nanpercentile(b, 97.5)}
    if n != rif:
        d = b - boot[rif]
        r.update({"Δ vs TF-IDF": tutto.loc[n, "media sugli orientamenti"] - tutto.loc[rif, "media sugli orientamenti"],
                  "Δ 2.5%": np.nanpercentile(d, 2.5), "Δ 97.5%": np.nanpercentile(d, 97.5), "P(Δ>0)": float(np.nanmean(d > 0))})
    righe[n] = r
iv = pd.DataFrame(righe).T
iv.to_csv(RESULTS / "03_genere_intervalli.csv")
print("riferimento:", rif)
iv.round(3)

riferimento: feature_genere_tuned


,media sugli orientamenti,2.5%,97.5%,Δ vs TF-IDF,Δ 2.5%,Δ 97.5%,P(Δ>0)
bge_genere,0.502,0.445,0.565,-0.091,-0.173,-0.010,0.016
e5_genere,0.569,0.494,0.664,-0.024,-0.079,0.045,0.258
feature_genere,0.618,0.546,0.737,0.025,-0.014,0.119,0.868
feature_genere_senza_nomi,0.589,0.526,0.669,-0.004,-0.031,0.029,0.406
feature_genere_solo_funzione,0.618,0.550,0.740,0.025,-0.015,0.122,0.869
feature_genere_tuned,0.593,0.526,0.668,NaN,NaN,NaN,NaN
modernbert_base_genere_s1,0.533,0.462,0.629,-0.060,-0.125,0.014,0.059
modernbert_base_genere_s42,0.579,0.506,0.699,-0.014,-0.087,0.093,0.408
modernbert_base_genere_ensemble2,0.561,0.497,0.645,-0.032,-0.091,0.034,0.176
combinazione_tfidf_modernbert,0.590,0.530,0.669,-0.002,-0.039,0.037,0.498
